# Evaluating HGLM's segmentation objective

Does HGLM's segmentation objective (removing per-voxel residuals first) improve upon Ward's?  By how much?

Approach:
- sample `n` effect extents
- for each, impose effect at multiple `f_ratio`
- segment according to HGLM & Ward
- report the max F1 score across segmentation for each

In [ ]:
import hglm
import numpy as np

n_jobs = -1
n_repeat = 10
radius = 4
effect_perc = .2
f_ratio = np.logspace(np.log10(.001), np.log10(.05), 9)

# load human connectome project data
folder = '/home/matt/Dropbox/pnl_hglm/data/hcp100_lowres/image'
exp_hcp = hglm.experiment.ExperimentImageOnly.from_search(
    folder=folder,
    sbj_regex=r'[\d]{6}',
    img_glob_dict={'FA': '*_FA.nii.gz',
                   'MD': '*_MD.nii.gz'})
exp = exp_hcp.sample_x(a=2, seed=0, add_bias=True)

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd
from itertools import product

def process_seed_f(seed, f, radius, exp, effect_perc):
    # trim experiment
    extenter = hglm.effect.ExtenterSphere(radius=radius)
    mask = extenter(mask_idx=exp.mask_idx, seed=seed, contiguous=True)
    exp_masked = exp.apply_mask(mask)

    # sample extent
    n = exp_masked.y.shape[2] * effect_perc
    extenter = hglm.effect.ExtenterMinVar(n=n)
    mask_target = extenter(y=exp_masked.y,
                           mask_idx=exp_masked.mask_idx,
                           seed=seed)

    # impose effect
    _exp, effect = exp_masked.impose_effect(mask=mask_target,
                                            seed=seed, f_ratio=f)
    
    # cluster (hglm & ward)
    maxf1_dict = dict()
    for mode in ('full', 'ward'):
        children = hglm.experiment.AnalysisHGLM.cluster(_exp, mode)
        f1 = hglm.graph.get_f1(mask=mask_target,
                               mask_idx=_exp.mask_idx,
                               children=children)
        maxf1_dict[mode] = f1.max()
        
    return dict(seed=seed, f_ratio=f, f1max_hglm=maxf1_dict['full'], f1max_ward=maxf1_dict['ward'])

param_grid = list(product(range(int(n_repeat)), f_ratio))

results = Parallel(n_jobs=n_jobs)(
    delayed(process_seed_f)(seed, f, radius, exp, effect_perc)
    for seed, f in tqdm(param_grid, desc='experiment (per seed-stat)')
)


df = pd.DataFrame(results)

In [ ]:
import matplotlib.pyplot as plt

# Assumes your DataFrame is called `df`
# df = pd.read_csv(...)  # Load your data if needed

# Set up the plot
plt.figure(figsize=(10, 6))

# Colors for methods
colors = {
    'HGLM': 'tab:blue',
    'Ward': 'tab:orange'
}

# Plot thin lines: one per seed per method (no legend)
for seed, group in df.groupby('seed'):
    group_sorted = group.sort_values('f_ratio')
    plt.plot(group_sorted['f_ratio'], group_sorted['f1max_hglm'],
             color=colors['HGLM'], alpha=0.3, linewidth=1)
    plt.plot(group_sorted['f_ratio'], group_sorted['f1max_ward'],
             color=colors['Ward'], alpha=0.3, linewidth=1)

# Plot average lines: mean across seeds per method
avg_df = df.groupby('f_ratio', as_index=False)[['f1max_hglm', 'f1max_ward']].mean()

plt.plot(avg_df['f_ratio'], avg_df['f1max_hglm'],
         label='HGLM (avg)', color=colors['HGLM'], linewidth=3)

plt.plot(avg_df['f_ratio'], avg_df['f1max_ward'],
         label='Ward (avg)', color=colors['Ward'], linewidth=3)

# Format plot
plt.xscale('log')
plt.xlabel('f_ratio')
plt.ylabel('F1 Score')
plt.title('F1 Score vs F-ratio per Seed and Averaged')
plt.legend(title='Method')
plt.tight_layout()
plt.show()